### Student Information
- **Name:** Muhammad Ahmad
- **Roll No:** 34801
- **Subject:** Machine Learning
- **Class:** BS - AI (Semester-05)

---

# BSAI • MACHINE LEARNING LAB
## Support Vector Machine (SVM) Lab Manual | Mr. Soban Ahmad
### Practical: Classifying Emails Using Two Features

| Component | Description |
| :--- | :--- |
| **Lab** | W6 – Support Vector Machine (SVM) |
| **Dataset** | Synthetic email dataset – 240 rows, 6 columns |
| **Model** | Linear SVM – `SVC(kernel="linear", C=1.0)` |
| **Input features** | `suspicious_links`, `promotional_words` |
| **Target** | `class_name` (`legitimate` / `spam`) |
| **Preprocessing** | `StandardScaler` (fitted on the training data only) |
| **Split** | 80/20 stratified Train-Test (192 / 48 rows), `random_state=42` |
| **Evaluation** | Accuracy, confusion matrix, classification report, decision-boundary plot |
| **Libraries** | pandas, NumPy, Matplotlib, scikit-learn |

**Basis:** This notebook follows the W6 Support Vector Machine lab manual prepared by Mr. Soban Ahmad for BSAI students; the original code and steps are preserved, and the student tasks and lab-report answers are added.

---

### Prerequisites & Asset Setup
- **Required File:** `SVM_Synthetic_Email_Dataset.csv` (the lab manual refers to the same file as `svm_synthetic_emails.csv`).
- **Repository Path:** This file is located in the repository under `Lab-06/assets/SVM_Synthetic_Email_Dataset.csv` (or `./assets/SVM_Synthetic_Email_Dataset.csv`). Ensure the CSV is in the same working directory as this notebook, or in its `assets/` folder, before running Step 2.
- **Google Colab:** if the file is not found, Step 2 opens the upload dialog so you can select the CSV.
- **Packages:** `pip install pandas numpy matplotlib scikit-learn` (already included in Google Colab).

## 1. Lab Objectives
- Load and inspect a synthetic CSV dataset containing email features.
- Separate the input features from the target class.
- Split the data into training and testing sets.
- Scale numeric features before training an SVM.
- Train a linear Support Vector Machine classifier.
- Identify support vectors, plot the decision boundary, and evaluate predictions.

## 2. Problem Statement
An email dataset contains two input features, `suspicious_links` and `promotional_words`, and a target class named `class_name`. The goal is to predict whether an email is **legitimate** or **spam**. This is a **classification** task because the target contains categories.

## 3. Dataset Plan: How We Solve the Dataset Problem
Use the supplied synthetic CSV file for this practical. It contains **240 rows and six columns**. The same file can be used by the whole class, so results are reproducible.

| Column | Role | Possible values |
| :--- | :--- | :--- |
| `suspicious_links` | Input feature | Integer count (0–8) |
| `promotional_words` | Input feature | Integer count (0–10) |
| `class_name` | Target label | `legitimate`, `spam` |

The columns `email_id`, `label`, and `borderline_case` are supporting information. For this lab, use **only** `suspicious_links` and `promotional_words` as the input features and `class_name` as the target. The dataset is synthetic and is intended for learning, not for real email filtering.

## 4. Software and Library
Use Python in Google Colab, Jupyter Notebook, or another Python environment. Required libraries: pandas, NumPy, Matplotlib, and scikit-learn.

```bash
pip install pandas numpy matplotlib scikit-learn
```

Google Colab usually includes these libraries already.

## 5. Practical Steps
### Step 1 — Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

`pandas` handles the dataset, `train_test_split` creates training and testing sets, `StandardScaler` scales features, and `SVC` trains the SVM classifier.

### Step 2 — Load the CSV file
Upload the CSV when Colab asks you to choose a file. The dataset has 240 rows and 6 columns; `value_counts()` shows the number of examples in each class.

> **Note:** the original manual uses `files.upload()` and the file name `svm_synthetic_emails.csv`. The loader below first looks for the CSV next to the notebook and in `./assets/` (under either file name) and falls back to the Colab upload dialog only if it is not found.

In [ ]:
import os

CSV_CANDIDATES = [
    "SVM_Synthetic_Email_Dataset.csv",
    "svm_synthetic_emails.csv",                                  # name used in the lab manual
    os.path.join("assets", "SVM_Synthetic_Email_Dataset.csv"),
    os.path.join("assets", "svm_synthetic_emails.csv"),
]
csv_path = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()                                # Select the dataset CSV
        csv_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "Dataset CSV not found. Place SVM_Synthetic_Email_Dataset.csv next to this "
            "notebook or in ./assets/ and run this cell again."
        )

df = pd.read_csv(csv_path)
print("Loaded file:", csv_path)
display(df.head())
print("Dataset shape:", df.shape)
print(df["class_name"].value_counts())

**Reference result:** the dataset has **240 rows × 6 columns** (`email_id`, `suspicious_links`, `promotional_words`, `label`, `class_name`, `borderline_case`) and is perfectly balanced: **120 legitimate** and **120 spam** emails.

### Step 3 — Separate features (X) and target (y)
`X` contains the two input features and `y` contains the target class. Do **not** include the target, `email_id`, or metadata columns in `X`.

In [ ]:
X = df[["suspicious_links", "promotional_words"]]
y = df["class_name"]

print(X.head())
print(y.head())

### Step 4 — Split into training and testing data
The model trains on 192 rows and is tested on 48. `random_state=42` makes the split reproducible, and `stratify=y` preserves the class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

**Reference result:** 192 training rows (96 legitimate + 96 spam) and 48 testing rows (24 legitimate + 24 spam).

### Step 5 — Scale the features
SVM depends on distances, so features must be scaled. Fit the scaler on the **training data only**, then use it to transform the test data. Standardisation maps each feature to zero mean and unit variance:

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Step 6 — Train a linear SVM model
A linear SVM learns a straight decision boundary in this two-feature space. **Support vectors** are the training examples that help determine the boundary and margin. **C** controls the trade-off between a wider margin and classification errors on the training data.

In [ ]:
model = SVC(kernel="linear", C=1.0)
model.fit(X_train_scaled, y_train)

print("Support vectors per class:", model.n_support_)
print("Total support vectors:", len(model.support_vectors_))

**Reference result:** support vectors per class = `[33 32]` (class order: `legitimate`, `spam`), i.e. **65 support vectors** out of 192 training rows.

### Step 7 — Predict and evaluate the model
Accuracy is the fraction of test examples predicted correctly. The confusion matrix compares the actual and predicted classes. Report the value produced by **your run** rather than assuming that every run or dataset must have a particular score.

In [ ]:
y_pred = model.predict(X_test_scaled)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Confusion matrix (legitimate, spam):")
print(confusion_matrix(y_test, y_pred,
                       labels=["legitimate", "spam"]))
print(classification_report(y_test, y_pred, zero_division=0))

**Reference result (this dataset, `random_state=42`):**

| Actual / Predicted | legitimate | spam |
| :--- | :--- | :--- |
| **legitimate** | 18 | 6 |
| **spam** | 2 | 22 |

| Class / Metric | Precision | Recall | F1 | Support |
| :--- | :--- | :--- | :--- | :--- |
| **legitimate** | 0.90 | 0.75 | 0.82 | 24 |
| **spam** | 0.79 | 0.92 | 0.85 | 24 |
| **Accuracy** | | | 0.83 | 48 |
| **Macro avg** | 0.84 | 0.83 | 0.83 | 48 |
| **Weighted avg** | 0.84 | 0.83 | 0.83 | 48 |

Test accuracy = $\frac{18 + 22}{48} = 0.833$. The model leans towards flagging emails as spam: 6 legitimate emails were wrongly flagged (recall 0.75 on `legitimate`), while only 2 spam emails slipped through (recall 0.92 on `spam`).

### Step 8 — Plot the decision boundary and support vectors
The **solid contour** is the decision boundary (score 0). The **dashed contours** indicate the margin (scores −1 and +1). The **outlined points** are support vectors. The axes show standardized feature values, so some values can be negative.

In [ ]:
plt.figure(figsize=(8, 5))
for class_name in ["legitimate", "spam"]:
    mask = (y_train == class_name).to_numpy()
    plt.scatter(X_train_scaled[mask, 0], X_train_scaled[mask, 1],
                edgecolor="k", label=class_name)

xx, yy = np.meshgrid(
    np.linspace(X_train_scaled[:, 0].min()-1, X_train_scaled[:, 0].max()+1, 250),
    np.linspace(X_train_scaled[:, 1].min()-1, X_train_scaled[:, 1].max()+1, 250)
)
Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"])
plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
            s=140, facecolors="none", edgecolors="black", label="Support vectors")

plt.xlabel("Standardized suspicious links")
plt.ylabel("Standardized promotional words")
plt.title("Linear SVM: Decision Boundary and Margin")
plt.legend()
plt.show()

**Reading the plot:** because both features are integer counts, many emails share exactly the same coordinates and the dots overlap (the 192 training rows contain only 59 distinct points). The solid line separates the region predicted *spam* (high suspicious links and promotional words) from the region predicted *legitimate*; points between the dashed lines or on the wrong side of the solid line are the ones that become support vectors.

### Step 9 — Predict a new email
The new example must contain the same feature columns as `X`, with the same column names and order. The output will be `legitimate` or `spam`. The classification is a model prediction, not a guarantee that the email is actually spam.

In [ ]:
new_email = pd.DataFrame([{
    "suspicious_links": 6,
    "promotional_words": 4
}])

prediction = model.predict(scaler.transform(new_email))
print("Predicted class:", prediction[0])

**Reference result:** an email with 6 suspicious links and 4 promotional words is predicted as **spam**.

## 6. Student Tasks

### Task 1: Predict a different new email
Change the new email to `suspicious_links = 2` and `promotional_words = 1`. Record the predicted class.

In [ ]:
new_email_t1 = pd.DataFrame([{
    "suspicious_links": 2,
    "promotional_words": 1
}])

prediction_t1 = model.predict(scaler.transform(new_email_t1))
print("Predicted class for (2 links, 1 promotional word):", prediction_t1[0])

**Answer (Task 1):** the predicted class is **legitimate**. Compared with the Step 9 email (6 links, 4 promotional words → spam), both feature counts are low, so the point falls on the legitimate side of the decision boundary.

### Task 2: Change C (0.1, 1.0, 10)
Change `C` from 1.0 to 0.1, then to 10. Compare test accuracy and the number of support vectors.

In [ ]:
rows = []
for C_value in [0.1, 1.0, 10]:
    svm_c = SVC(kernel="linear", C=C_value)
    svm_c.fit(X_train_scaled, y_train)
    rows.append({
        "C": C_value,
        "Test accuracy": round(accuracy_score(y_test, svm_c.predict(X_test_scaled)), 3),
        "Train accuracy": round(svm_c.score(X_train_scaled, y_train), 3),
        "Support vectors (legitimate, spam)": tuple(int(n) for n in svm_c.n_support_),
        "Total support vectors": len(svm_c.support_vectors_)
    })

task2_df = pd.DataFrame(rows)
display(task2_df)

**Answer (Task 2):**

| C | Test accuracy | Support vectors (legitimate, spam) | Total |
| :--- | :--- | :--- | :--- |
| 0.1 | 0.854 (41/48) | (41, 41) | 82 |
| 1.0 | 0.833 (40/48) | (33, 32) | 65 |
| 10 | 0.833 (40/48) | (31, 32) | 63 |

1. **Support vectors decrease as C grows** (82 → 65 → 63). A small C allows a wide margin that tolerates many points inside it or on the wrong side, so many points become support vectors. A large C penalises margin violations heavily, giving a narrower margin and fewer support vectors.
2. **Test accuracy barely changes.** The difference between 0.854 and 0.833 is **one email** out of 48, so on such a small test set it is not evidence that C = 0.1 is truly better. The data is noisy (borderline emails overlap), so the boundary is already close to the best straight line for every C.

### Task 3: Change the kernel from `linear` to `rbf`
Compare its test accuracy with the linear model.

In [ ]:
rows = []
for kernel_name in ["linear", "rbf"]:
    svm_k = SVC(kernel=kernel_name, C=1.0)
    svm_k.fit(X_train_scaled, y_train)
    rows.append({
        "Kernel": kernel_name,
        "Test accuracy": round(accuracy_score(y_test, svm_k.predict(X_test_scaled)), 3),
        "Train accuracy": round(svm_k.score(X_train_scaled, y_train), 3),
        "Support vectors (legitimate, spam)": tuple(int(n) for n in svm_k.n_support_),
        "Total support vectors": len(svm_k.support_vectors_)
    })

task3_df = pd.DataFrame(rows)
display(task3_df)

**Answer (Task 3):**

| Kernel | Test accuracy | Train accuracy | Total support vectors |
| :--- | :--- | :--- | :--- |
| linear | 0.833 | 0.870 | 65 |
| rbf | 0.833 | 0.875 | 70 |

The RBF kernel can bend the boundary, but it gives the **same test accuracy (0.833)** and only a negligible gain in training accuracy. The two classes in this dataset are essentially separated by a straight line; the remaining errors come from overlapping (borderline) emails that no smooth boundary can separate, not from a curved class shape. A linear SVM is therefore sufficient and simpler.

### Task 4: Use only `suspicious_links`
Train the model using only `suspicious_links`. Compare the result with the model that uses both features. A separate scaler is fitted for this single feature.

In [ ]:
X_train_1 = X_train[["suspicious_links"]]
X_test_1 = X_test[["suspicious_links"]]

scaler_1 = StandardScaler()
X_train_1_scaled = scaler_1.fit_transform(X_train_1)
X_test_1_scaled = scaler_1.transform(X_test_1)

model_1 = SVC(kernel="linear", C=1.0)
model_1.fit(X_train_1_scaled, y_train)
y_pred_1 = model_1.predict(X_test_1_scaled)

print("Model with ONE feature (suspicious_links)")
print("Test accuracy:", round(accuracy_score(y_test, y_pred_1), 3))
print("Confusion matrix (legitimate, spam):")
print(confusion_matrix(y_test, y_pred_1, labels=["legitimate", "spam"]))
print("Support vectors per class:", model_1.n_support_)

print("\nComparison")
print("Two features test accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("One feature  test accuracy:", round(accuracy_score(y_test, y_pred_1), 3))

**Answer (Task 4):**

| Model | Test accuracy | Confusion matrix (legit, spam) | Support vectors |
| :--- | :--- | :--- | :--- |
| Both features | 0.833 | [[18, 6], [2, 22]] | 65 |
| `suspicious_links` only | 0.750 | [[12, 12], [0, 24]] | 82 |

Dropping `promotional_words` **lowers accuracy by 8.3 percentage points**: the single-feature model makes 12 mistakes on the 48 test emails instead of 8, i.e. 4 more. With a single feature the boundary is just one threshold on the link count: every spam email is caught (recall 1.00), but half of the legitimate emails are flagged as spam. `promotional_words` carries extra information that separates the legitimate emails with a few links from real spam, so both features should be used.

### Task 5: Explain the plot
Look at the plot from Step 8. Explain what the solid line, dashed lines, and outlined points represent.

**Answer (Task 5):**
1. **Solid line:** the **decision boundary**, where the SVM score `decision_function(x)` equals 0. Points on one side are predicted `spam`, points on the other side `legitimate`.
2. **Dashed lines:** the **margin boundaries**, where the score equals −1 and +1. The strip between them is the margin; a linear SVM chooses the line that makes this strip as wide as possible while keeping margin violations small (controlled by C).
3. **Outlined points:** the **support vectors**, i.e. the training emails that lie on the dashed lines, inside the margin, or on the wrong side of the boundary. Only these points determine the position of the line; removing any other training point would not change the model.

## 7. Questions for Lab Report

**6. What are the input features and target column in this dataset?**
- **Input features:** `suspicious_links` (integer count, 0–8) and `promotional_words` (integer count, 0–10).
- **Target column:** `class_name` with the values `legitimate` and `spam`.
- `email_id`, `label` and `borderline_case` are supporting columns and are not used as features. `label` is only a numeric copy of `class_name` (0 = legitimate, 1 = spam); using it as a feature would leak the answer.

**7. Why do we split the data into training and testing sets?**

The model must be judged on emails it has **never seen**. If it were tested on the same rows it learned from, the score would measure memorisation rather than generalisation. The split gives 192 rows for learning and 48 rows for an honest evaluation; `stratify=y` keeps the 50/50 class balance in both parts and `random_state=42` makes the result reproducible.

**8. Why is feature scaling useful for an SVM?**

An SVM maximises a margin that is measured with distances, so a feature with a larger numeric range would dominate the distance and pull the boundary towards itself. Standardisation gives every feature zero mean and unit variance so each contributes fairly. The scaler is fitted on the **training data only** and then applied to the test data and to any new email, otherwise information from the test set would leak into training. (In this dataset the two features have similar ranges, 0–8 and 0–10, so the effect is modest, but scaling is essential in general and for the RBF kernel.)

**9. What is a support vector, and why does it matter?**

A support vector is a training example that lies on the margin boundary, inside the margin, or on the wrong side of the decision boundary. The SVM solution depends **only** on these points: all other training points could be removed without changing the boundary. In this lab the linear model with C = 1.0 has **65 support vectors** (33 legitimate, 32 spam) out of 192 training rows.

**10. What does the margin represent in the SVM plot?**

The margin is the strip between the two dashed lines (scores −1 and +1) around the solid decision boundary. A wider margin means the classes are separated with more "safety distance", which usually generalises better. In scaled units its width is $\frac{2}{\lVert w \rVert}$, where $w$ is the weight vector of the linear model. The cell below computes it.

In [ ]:
w = model.coef_[0]
margin_width = 2 / np.linalg.norm(w)

print("Weight vector w (scaled features):", np.round(w, 3))
print("Intercept b:", np.round(model.intercept_, 3))
print("Margin width 2/||w||:", round(margin_width, 3), "(in standardized units)")

**Reference result:** $w \approx (1.575,\ 0.902)$, so the margin width is $\frac{2}{\lVert w \rVert} \approx 1.10$ standardized units. Both weights are positive, meaning that more suspicious links and more promotional words push an email towards `spam`; `suspicious_links` has the larger weight, so it influences the decision more.

**11. What was your test accuracy? Explain what the value means.**

In [ ]:
final_accuracy = accuracy_score(y_test, y_pred)
n_correct = int((y_pred == y_test.to_numpy()).sum())

print(f"Test accuracy: {final_accuracy:.3f} ({final_accuracy * 100:.1f}%)")
print(f"Correct predictions: {n_correct} of {len(y_test)} test emails")
print(f"Wrong predictions  : {len(y_test) - n_correct}")

**Answer (Q11):** the test accuracy is **0.833 (83.3%)**: the linear SVM classified **40 of the 48** unseen test emails correctly and made 8 mistakes (6 legitimate emails flagged as spam, 2 spam emails missed). Because the classes are perfectly balanced (24 / 24), accuracy is a fair summary here, and always predicting one class would give only 50%.

Caveats: the value comes from **one** train/test split of a **synthetic** dataset, and a single test email changes accuracy by about 2.1 percentage points, so it should not be read as proof of real-world spam-filtering performance. Most of the errors fall on emails the dataset marks as borderline (see the check below).

In [ ]:
# Extra check (analysis only - borderline_case is NOT used as a model feature)
wrong_mask = (y_pred != y_test.to_numpy())
wrong_idx = y_test.index[wrong_mask]

print("Misclassified test emails:", len(wrong_idx))
print("  of which marked borderline_case:", int(df.loc[wrong_idx, "borderline_case"].sum()))
print("Borderline emails in the test set:", int(df.loc[y_test.index, "borderline_case"].sum()), "of", len(y_test))

**Reference result:** 6 of the 8 misclassified emails are borderline cases, although borderline emails make up only 11 of the 48 test rows. The errors therefore sit exactly where the two classes overlap, which explains why a more flexible kernel or a different C hardly helps.

**12. How did changing C or the kernel affect the results? Support your answer with your outputs.**
- **Changing C (Task 2):** smaller C widened the margin and increased the number of support vectors (C = 0.1: 82, C = 1.0: 65, C = 10: 63). Test accuracy moved only between 0.854 and 0.833, a difference of a single email out of 48.
- **Changing the kernel (Task 3):** RBF gave the same test accuracy as the linear kernel (0.833 vs 0.833) with slightly more support vectors (70 vs 65). The extra flexibility did not help because the classes are already almost linearly separable and the errors come from overlapping borderline emails.
- **Conclusion:** for this dataset the choice of C and kernel has little effect on accuracy; the linear kernel with C = 1.0 is a simple and sufficient model.

## 8. Important Notes and Common Errors
- Use only `suspicious_links` and `promotional_words` as the input features for this practical.
- Fit `StandardScaler` on the training data only, then transform the test data using the same scaler.
- Scale new email features with the fitted scaler before prediction.
- Use the same feature names and order when predicting a new example.
- A synthetic dataset is for learning the workflow; results do not prove real-world spam-filtering performance.
- A high score on one train/test split does not guarantee the same performance on unseen future emails.

## 9. Expected Submission
Submit your Colab notebook, test accuracy, confusion matrix, decision-boundary plot, prediction for the new email, and brief answers to the student tasks and lab-report questions.

- [x] Notebook with all code outputs and the decision-boundary plot rendered
- [x] Test accuracy and confusion matrix (Step 7)
- [x] Decision-boundary plot (Step 8)
- [x] Prediction for the new email (Step 9)
- [x] Answers to the student tasks (Section 6) and the lab-report questions (Section 7)